# 03 - Classificação de Texto com BERTugues e Random Forest

Neste notebook, vamos demonstrar como utilizar o modelo `ricardoz/BERTugues-base-portuguese-cased` da biblioteca `transformers` do Hugging Face apenas como um extrator de características (embeddings). 

Em vez de realizar o *fine-tuning* (ajuste fino) de todo o modelo BERT para a tarefa de classificação, nós vamos:
1. Passar os textos pelo modelo BERTugues.
2. Extrair os *embeddings* (vetores de representação) gerados.
3. Treinar um classificador tradicional (`RandomForestClassifier` do `scikit-learn`) usando esses embeddings como *features*.

**Nesta versão, comparamos duas estratégias de extração (Pooling): CLS vs Mean.**

## 1. Importando as bibliotecas necessárias

In [1]:
# !pip install transformers torch scikit-learn pandas numpy datasets

import torch
import pandas as pd
import numpy as np
import os
import urllib.request
from transformers import AutoTokenizer, AutoModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm.auto import tqdm

## 2. Carregando os Dados
Para obter nossa massa de testes sem depender de APIs instáveis, o script fará o download direto do CSV oficial do **B2W-Reviews** hospedado no GitHub e salvará em uma pasta `datasets` local do seu projeto (como cache).

In [2]:
# ================= CONFIGURAÇÃO =================
tamanho_amostra = 50000
# ================================================

url_csv = "https://raw.githubusercontent.com/b2wdigital/b2w-reviews01/master/B2W-Reviews01.csv"
pasta_dataset = "datasets"
arquivo_csv_local = f"{pasta_dataset}/B2W-Reviews01.csv"

os.makedirs(pasta_dataset, exist_ok=True)

if not os.path.exists(arquivo_csv_local):
    print(f"Baixando CSV para {arquivo_csv_local}...")
    urllib.request.urlretrieve(url_csv, arquivo_csv_local)
else:
    print("Cache detectado!")

df_raw = pd.read_csv(arquivo_csv_local, sep=',', low_memory=False, on_bad_lines='skip')
df_limpo = df_raw.dropna(subset=['review_text', 'overall_rating'])

if tamanho_amostra > len(df_limpo):
    tamanho_amostra = len(df_limpo)
    
df = df_limpo.sample(n=tamanho_amostra, random_state=42)
df['sentimento'] = pd.to_numeric(df['overall_rating'], errors='coerce').apply(lambda x: 1 if x > 3 else 0)
df['texto'] = df['review_text'].astype(str)
df = df[['texto', 'sentimento']]

display(df.head())
print(f"Total de registros: {len(df)}")

Cache detectado!


,texto,sentimento
127417,Suporte muito bom. Veio com varias opções de f...,1
13428,"livro muito bom, explica como interagir com o...",1
4711,"O aparelho é bom, mas os produtos que o acompa...",0
11496,Não veio como o combinado. Veio outra imagem. ...,0
24767,Tive um LG K10 anteriormente e gostava muito d...,1


Total de registros: 50000


## 3. Carregando o Tokenizador e o Modelo BERTugues

In [3]:
model_name = "ricardoz/BERTugues-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval();

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ricardoz/BERTugues-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4. Extraindo os Embeddings (CLS e Mean) com Otimização em Batch

Processar as frases uma por uma é lento. Vamos utilizar **Batching** (lotes) para aproveitar melhor o paralelismo da CPU ou GPU.

In [4]:
def extract_embeddings_batch(texts, batch_size=32):
    """
    Extrai embeddings em lotes para maior performance.
    """
    all_cls_embeddings = []
    all_mean_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extraindo Features em Batch"):
        batch = texts[i : i + batch_size].tolist()
        
        # Tokeniza o lote
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {key: value.to(device) for key, value in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        last_hidden_state = outputs.last_hidden_state
        
        # 1. Estratégia CLS (primeiro token de cada sequência no batch)
        cls_batch = last_hidden_state[:, 0, :].cpu().numpy()
        all_cls_embeddings.append(cls_batch)
        
        # 2. Estratégia Mean (média ponderada pela attention mask para ignorar PADs)
        attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * attention_mask, 1)
        sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
        mean_batch = (sum_embeddings / sum_mask).cpu().numpy()
        all_mean_embeddings.append(mean_batch)
        
    # Concatena todos os lotes em matrizes finais
    return np.vstack(all_cls_embeddings), np.vstack(all_mean_embeddings)

# Executa a extração otimizada
X_cls, X_mean = extract_embeddings_batch(df['texto'].values, batch_size=32)
y = df['sentimento'].values

print(f"\nFormato final X_cls: {X_cls.shape}")
print(f"Formato final X_mean: {X_mean.shape}")

Extraindo Features em Batch:   0%|          | 0/1563 [00:00<?, ?it/s]


Formato final X_cls: (50000, 768)
Formato final X_mean: (50000, 768)


## 5. Treinando Modelos para Comparação

In [5]:
# Divisão Treino/Teste para CLS
X_train_c, X_test_c, y_train, y_test = train_test_split(X_cls, y, test_size=0.3, random_state=42)
# Divisão Treino/Teste para Mean
X_train_m, X_test_m, _, _ = train_test_split(X_mean, y, test_size=0.3, random_state=42)

rf_cls = RandomForestClassifier(n_estimators=100, random_state=42)
rf_mean = RandomForestClassifier(n_estimators=100, random_state=42)

print("Treinando modelo CLS...")
rf_cls.fit(X_train_c, y_train)

print("Treinando modelo Mean...")
rf_mean.fit(X_train_m, y_train)

Treinando modelo CLS...
Treinando modelo Mean...


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

## 6. Avaliação Comparativa

In [6]:
y_pred_c = rf_cls.predict(X_test_c)
y_pred_m = rf_mean.predict(X_test_m)

print(f"Acurácia CLS: {accuracy_score(y_test, y_pred_c):.4f}")
print(f"Acurácia Mean: {accuracy_score(y_test, y_pred_m):.4f}")

print("\n--- Detalhes CLS ---")
print(classification_report(y_test, y_pred_c))

print("\n--- Detalhes Mean ---")
print(classification_report(y_test, y_pred_m))

Acurácia CLS: 0.8621
Acurácia Mean: 0.8665

--- Detalhes CLS ---
              precision    recall  f1-score   support

           0       0.84      0.79      0.82      5858
           1       0.87      0.91      0.89      9142

    accuracy                           0.86     15000
   macro avg       0.86      0.85      0.85     15000
weighted avg       0.86      0.86      0.86     15000


--- Detalhes Mean ---
              precision    recall  f1-score   support

           0       0.85      0.80      0.82      5858
           1       0.88      0.91      0.89      9142

    accuracy                           0.87     15000
   macro avg       0.86      0.85      0.86     15000
weighted avg       0.87      0.87      0.87     15000



## 7. Prevendo em Novos Textos

In [7]:
novos = ["Adorei o produto!", "Péssima qualidade, ruim, não recomendo."]

# Reaproveitamos a função de batch para os novos textos (mesmo que sejam poucos)
c_new, m_new = extract_embeddings_batch(np.array(novos), batch_size=2)

for i, t in enumerate(novos):
    p_c = "Positivo" if rf_cls.predict([c_new[i]])[0] == 1 else "Negativo"
    p_m = "Positivo" if rf_mean.predict([m_new[i]])[0] == 1 else "Negativo"
    print(f"Texto: {t} | CLS: {p_c} | Mean: {p_m}")

Extraindo Features em Batch:   0%|          | 0/1 [00:00<?, ?it/s]

Texto: Adorei o produto! | CLS: Positivo | Mean: Positivo
Texto: Péssima qualidade, ruim, não recomendo. | CLS: Negativo | Mean: Negativo
